# BharatMatrix — cross-encoder fine-tuning

Fine-tunes a small, pretrained cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`,
~22M params) on BharatMatrix's own question/passage pairs — synthetic ones
generated from the app's documents, plus real ones logged from actual usage
(see `generate_synthetic_data.py` and `download_feedback.mjs` in this same
`cross-encoder/` folder), optionally mixed with a public finance-QA head start
(`build_public_dataset.py`, see `DATASETS.md`). Run `combine_and_split.py` to
get `train.jsonl` / `val.jsonl` before starting this notebook.

**Honest framing**: this is *fine-tuning* a pretrained checkpoint, not training
a cross-encoder from random weights. A from-scratch model would need millions
of labeled pairs and far more compute than a free Colab GPU gives you in one
session — fine-tuning an existing small cross-encoder on your own domain data
is the real, standard way a project this size gets "its own" model, and
`rerankWithFallback` in the app only ever needed *a* working model behind
`CROSS_ENCODER_URL`, not one trained from zero.

**Before you start**: Runtime → Change runtime type → GPU (the free T4 is enough
for a model this small).

In [ ]:
!pip install -q sentence-transformers==3.*

## 1. Upload the dataset

Run the cell below and pick `train.jsonl` and `val.jsonl` from your computer
(the files `combine_and_split.py` produced locally).

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # select train.jsonl and val.jsonl (and public_test.jsonl if you built it)
assert "train.jsonl" in uploaded and "val.jsonl" in uploaded, (
    "Upload both train.jsonl and val.jsonl (produced by combine_and_split.py)"
)
HAS_PUBLIC_TEST = os.path.exists("public_test.jsonl")
print("public_test.jsonl:", "found" if HAS_PUBLIC_TEST else "not uploaded (fine — that check is skipped)")


In [ ]:
import json
from sentence_transformers import InputExample

def load_examples(path):
    examples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            examples.append(
                InputExample(texts=[row["question"], row["passage"]], label=float(row["label"]))
            )
    return examples

train_examples = load_examples("train.jsonl")
val_examples = load_examples("val.jsonl")
print(f"{len(train_examples)} training pairs, {len(val_examples)} validation pairs")

## 2. Load the base model

`ms-marco-MiniLM-L-6-v2` is a well-known small cross-encoder already trained
for passage ranking (MS MARCO) — a sensible, honest starting point rather than
random weights. `num_labels=1` sets it up for binary relevance scoring
(sigmoid over one logit), matching the 0/1 labels in the dataset.

In [ ]:
from sentence_transformers import CrossEncoder

# max_length MUST match MAX_LENGTH in serve/app.py (default 256). The app's
# chunks are <= 900 characters (~225 tokens), so 256 holds a chunk + question,
# and it is faster than 384/512 on the CPU Space that serves this model.
MAX_LENGTH = 256
BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=MAX_LENGTH)


## 2b. A ranking metric that matches the job

The validation accuracy printed during training answers "is this single pair
relevant?". The app's real job is different: given a question, put the right
passage *above* the wrong ones. So this cell groups the rows by question and
reports two numbers:

* **Recall@1** — how often the top-ranked passage is a labelled-relevant one.
* **MRR** — mean of 1 / (rank of the first relevant passage).

It runs on the untrained base model first, so the "after" numbers have
something honest to be compared with. A model that doesn't beat its own
starting point on these has not learned anything useful, whatever the loss did.


In [ ]:
from collections import defaultdict

def ranking_metrics(model, path):
    groups = defaultdict(list)  # question -> [(passage, label)]
    with open(path, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            groups[r["question"]].append((r["passage"], int(r["label"])))
    hits = rr = n = 0
    for q, items in groups.items():
        if not any(l == 1 for _, l in items) or all(l == 1 for _, l in items):
            continue  # need at least one positive and one negative to rank
        scores = model.predict([(q, p) for p, _ in items], show_progress_bar=False)
        order = sorted(range(len(items)), key=lambda i: scores[i], reverse=True)
        first_pos = next(rank for rank, i in enumerate(order, start=1) if items[i][1] == 1)
        hits += first_pos == 1
        rr += 1.0 / first_pos
        n += 1
    return {"questions": n, "recall@1": hits / max(n, 1), "mrr": rr / max(n, 1)}

def report(tag, m):
    for name, path in (("val", "val.jsonl"), ("public_test", "public_test.jsonl")):
        if name == "public_test" and not HAS_PUBLIC_TEST:
            continue
        r = ranking_metrics(m, path)
        print(f"{tag:>7} | {name:<11} | {r['questions']:>5} questions | "
              f"Recall@1 {r['recall@1']:.3f} | MRR {r['mrr']:.3f}")

report("before", model)


## 3. Fine-tune

A handful of epochs over a dataset this size is plenty — cross-encoders
overfit fast on small data, which is also why `val_examples` (via the
evaluator below) matters more here than the final train loss number.

In [ ]:
import math
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

evaluator = CEBinaryClassificationEvaluator.from_input_examples(
    val_examples, name="bharatmatrix-val"
)

num_epochs = 4
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)

model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    evaluation_steps=max(1, len(train_dataloader) // 2),
    output_path="output/bharatmatrix-cross-encoder",
    show_progress_bar=True,
)

# Note: with an evaluator, fit() saves the BEST checkpoint (by validation
# score) into output_path, which is not necessarily the weights `model`
# holds after the last epoch. Everything below therefore reloads from disk.

### Did it actually improve?

Reloads the best saved checkpoint and re-runs the ranking metrics. Compare with
the "before" line above. If `public_test` improves but `val` doesn't (or the
reverse), that's informative too — `val` reflects your own documents,
`public_test` reflects general finance QA the model never trained on.

In [ ]:
best_model = CrossEncoder("output/bharatmatrix-cross-encoder", num_labels=1, max_length=MAX_LENGTH)
report("after", best_model)


## 4. Sanity-check it

Score a question against a few passages by hand before trusting it in
production — the sign of the score matters more than its exact magnitude
(this is a binary relevance score, not a calibrated probability).

In [ ]:
example_question = "What was the revenue in 2025?"
example_passages = [
    "Total revenue for fiscal year 2025 was 482 crore, up 14% year-over-year.",
    "The company's headquarters relocated to Bengaluru in March 2024.",
]
scores = best_model.predict([(example_question, p) for p in example_passages])
for score, passage in sorted(zip(scores, example_passages), reverse=True):
    print(f"{score:.3f}  {passage}")

## 5. Get the model out of Colab

Two ways to do it — pick one. **Pushing to the Hugging Face Hub (5a) is the
easier path**, since `cross-encoder/serve/app.py` can then load the model
directly by its Hub repo id, with nothing to manually copy into the Space.
(`save_to_hub` is deliberately not used: it isn't a method I could confirm exists
across sentence-transformers versions, whereas `huggingface_hub`'s `upload_folder`
is a stable, documented API.)

### 5a. Push to the Hugging Face Hub (recommended)

In [ ]:
from huggingface_hub import login

login()  # pastes in a Hugging Face access token (Settings -> Access Tokens on huggingface.co) — needs "Write" scope

In [ ]:
from huggingface_hub import HfApi

# Replace with your own Hugging Face username and whatever repo name you like.
HF_REPO_ID = "YOUR_HF_USERNAME/bharatmatrix-cross-encoder"

api = HfApi()
api.create_repo(HF_REPO_ID, exist_ok=True)  # private=True if you'd rather keep it unlisted
api.upload_folder(folder_path="output/bharatmatrix-cross-encoder", repo_id=HF_REPO_ID)
print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")
print("Set MODEL_PATH to this repo id in the Space settings — see cross-encoder/serve/README.md.")
print("A private repo also needs an HF_TOKEN secret on the Space so it can download the model.")


### 5b. Or just download the folder as a zip

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("bharatmatrix-cross-encoder", "zip", "output/bharatmatrix-cross-encoder")
files.download("bharatmatrix-cross-encoder.zip")